In [1]:
import sys, platform, importlib.metadata, json
print("AUDIT_KERNEL="+json.dumps({"python":sys.version,"executable":sys.executable,"platform":platform.platform(),"packages":dict(sorted((d.metadata["Name"],d.version) for d in importlib.metadata.distributions()))}))

AUDIT_KERNEL={"python": "3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]", "executable": "C:\\Users\\Miakhlov\\Documents\\GitHub\\Computational chemistry_course\\scripts\\environment\\.venv-chem\\Scripts\\python.exe", "platform": "Windows-10-10.0.19045-SP0", "packages": {"Jinja2": "3.1.6", "MarkupSafe": "3.0.4", "PyYAML": "6.0.3", "Pygments": "2.21.0", "Send2Trash": "2.2.0", "altair": "6.3.0", "anyio": "4.16.0", "argon2-cffi": "25.1.0", "argon2-cffi-bindings": "26.2.0", "arrow": "1.4.0", "asttokens": "3.0.2", "async-lru": "2.4.0", "attrs": "26.1.0", "babel": "2.18.0", "beautifulsoup4": "4.15.0", "bleach": "6.4.0", "certifi": "2026.10.23", "cffi": "2.1.1", "charset-normalizer": "3.5.2", "click": "8.5.1", "cloudpickle": "3.1.3", "colorama": "0.4.6", "comm": "0.2.3", "contourpy": "1.4.0", "cycler": "0.12.1", "debugpy": "1.8.22", "defusedxml": "0.7.1", "executing": "2.3.0", "fastjsonschema": "2.22.3", "fonttools": "4.68.0", "fqdn": "1.6.0", "h11": "0.16.0

# Разбор A. Технологическое решение

Маршрут с лучшим общим выходом может быть хуже по стоимости, отходам или контролю стереохимии. На A разбираем явно вымышленную таблицу routes.csv; её числа не относятся к промышленному синтезу истрадефиллина. B применяет баланс к условным маршрутам и формулирует требования к реальным данным процесса.

Рассчитайте Y и C_eff двух учебных маршрутов. Проверьте сценарий ±.05 абсолютного выхода каждой стадии и изменение рейтинга. Явно отделите неизвестные объёмы растворителей, стоимость очистки и риск стереоизомеров. Для B сформулируйте материальный баланс без выдуманных промышленных чисел.

In [2]:
from pathlib import Path
import sys,json,math
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'programm/course.json').is_file())
sys.path.insert(0,str(ROOT/'practicuum/shared'))
from course_support import DATA,compound,esol,workspace,write_json,init_dossier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
def publish_plot(fig,name):
    fig.savefig(OUT/name,dpi=120,bbox_inches='tight')
    display(fig)
    plt.close(fig)
OUT=workspace('M13-L02','case')


Сначала проверяем входы и единицы. Словари сохраняют названия полей вместе со значениями; список сохраняет соответствие записей. Выведите промежуточный результат перед дальнейшим анализом.

In [3]:
import csv
with (DATA/'routes.csv').open(encoding='utf-8',newline='') as f:rows=list(csv.DictReader(f))
values=[]
for r in rows:
    a,b,c=map(float,[r['yield1'],r['yield2'],r['assumed_cost']]);total=a*b
    values.append({'route':r['route'],'overall_yield':total,'cost_per_equivalent':c/total,
                   'lower_yield_scenario':(a-.05)*(b-.05)})
display(values)


[{'route': 'A',
  'overall_yield': 0.5599999999999999,
  'cost_per_equivalent': 1071.4285714285716,
  'lower_yield_scenario': 0.48749999999999993},
 {'route': 'B',
  'overall_yield': 0.855,
  'cost_per_equivalent': 877.1929824561404,
  'lower_yield_scenario': 0.7649999999999999}]

Теперь сравним модель с контрольным случаем. `assert` останавливает расчёт при нарушении контракта; это защита от ошибочного продолжения, а не оценка научного понимания.

In [4]:
fig,ax=plt.subplots();ax.bar([r['route'] for r in values],[r['cost_per_equivalent'] for r in values])
ax.set(xlabel='Fictional route',ylabel='Simulated cost / product equivalent');publish_plot(fig,'routes.png')


<Figure size 640x480 with 1 Axes>

Сохраним именно полученный результат. Не переносите этот вывод на другую химическую форму, мишень или экспериментальные условия без отдельной проверки.

In [5]:
result={'kind':'simulation','routes':values,'PMI':'NOT_CALCULABLE: missing masses','real_drug_manufacturing':'NOT_RUN'}


In [6]:
write_json(OUT/'result.json',result)
from dossier_records import append_record
receipt=append_record(init_dossier(),'M13-L02','A',result,ROOT,inputs=['practicuum/shared/data/routes.csv'])
print('Versioned evidence:',receipt)


Versioned evidence: C:\Users\Miakhlov\Documents\GitHub\Computational chemistry_course\practicuum\evidence\20261119-main-unification\notebooks\workspace\my_research\evidence\M13-L02\20261119T123405749869Z-A-6c968e6a44f74ba2b66b739a88dd59ee.json


Ни yield, ни cost surrogate не оценивают безопасность процесса. Научное решение включает область допустимых условий и запрос недостающих измерений. В финальном модуле соберём цепочку доказательств и укажем, что ещё не подтверждено.

Перед переходом ответьте: какие результаты получены вычислением, какие заданы условно, и какая независимая проверка ещё отсутствует?

# M13. Как наставник читает первичную работу

Пока следующая вычислительная задача ещё не выбрана, вернёмся к источнику: **Novozhilov et al. (2015). An improved kilogram-scale preparation of atorvastatin calcium**. [Локальный полный текст](../../../../../shared/data/source-responses/atorvastatin_process_verified.xml), [DOI](https://doi.org/10.1186/s13065-015-0082-7). Откройте **Scheme и Experimental финальных стадий**.

На примере статьи о ксантинах отделите выход отдельной стадии от заявления о полном маршруте.

Наставник начинает с вопроса «что здесь непосредственно наблюдали?». Найдите одно такое наблюдение и его место в источнике. Затем рядом запишите трактовку авторов и ещё одно возможное объяснение. Это учебный приём чтения, а не реконструкция мыслей авторов. Если численного ряда нет, вывод нельзя усиливать придуманными точками.

Перед вычислением назовите объект, условия, величину и единицы. После вычисления сравните ответ с исходным утверждением. При расхождении сначала проверьте форму вещества, основу концентрации и преобразования таблицы; только затем обсуждайте модель. Смысл операции — получить проверяемый аргумент для следующего решения досье.

Теперь перенесите приём на [самостоятельную мини-рецензию B](../steps/03_article_review.md). Для проверки собственного вывода используйте вопрос: **Можно ли назвать финальные превращения полной промышленной схемой синтеза из товарного сырья?**